In [0]:
import json, time
from confluent_kafka import Producer

producer = Producer(conf)

# Called once per event, after the broker confirms (or rejects) it.
def delivery_report(err, msg):
    if err is not None:
        print(f"  FAILED: {err}")
    else:
        key = msg.key().decode("utf-8")
        print(f"  delivered  key={key:8}  ->  partition {msg.partition()}  offset {msg.offset()}")

TOPIC = "post-engagement"

# 5 events, ALL for the same post -> same key -> should all land in ONE partition
events = [
    {"post_id": "post-1", "type": "view", "user_id": "u10", "ts": int(time.time())},
    {"post_id": "post-1", "type": "view", "user_id": "u11", "ts": int(time.time())},
    {"post_id": "post-1", "type": "like", "user_id": "u10", "ts": int(time.time())},
    {"post_id": "post-1", "type": "view", "user_id": "u12", "ts": int(time.time())},
    {"post_id": "post-1", "type": "like", "user_id": "u12", "ts": int(time.time())},
]

print("Producing 5 events for post-1 ...")
for e in events:
    producer.produce(
        topic=TOPIC,
        key=e["post_id"],              # the KEY -> decides the partition
        value=json.dumps(e),           # the VALUE -> the event body, as JSON text
        callback=delivery_report,      # called when the broker confirms this event
    )
    producer.poll(0)                   # give the client a moment to fire callbacks

producer.flush()                       # wait until ALL events are confirmed before finishing
print("Done.")